# Data Lake Preprocessing with PySpark

This notebook builds a small Bronze/Silver data lake for `sales.csv`. The Bronze copy is kept untouched; the Silver layer contains only basic quality fixes required by the assignment.

## 1. Create the data-lake folders

The Bronze layer stores the raw file. The Silver layer stores the cleaned result.

In [ ]:
from pathlib import Path

BRONZE = Path('data_lake/bronze')
SILVER = Path('data_lake/silver/preprocessed_sales')
BRONZE.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)

print('Bronze:', BRONZE)
print('Silver:', SILVER)


## 2. Put `sales.csv` in Bronze

Run this cell in Colab and upload the supplied dataset. The uploaded file is copied to Bronze without editing its contents.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()
if 'sales.csv' not in uploaded:
    raise ValueError('Please upload a file named sales.csv')

bronze_file = BRONZE / 'sales.csv'
shutil.copyfile('sales.csv', bronze_file)
print(f'Raw file stored at: {bronze_file}')


## 3. Install and start Spark

In [ ]:
!pip -q install pyspark


In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName('SalesDataLakeSilver')
         .getOrCreate())

raw = spark.read.option('header', True).csv(str(BRONZE / 'sales.csv'))
print('Raw records:', raw.count())
raw.printSchema()
raw.show(5, truncate=False)


## 4. Profile data-quality issues

The checks below look for exact duplicates, blanks, numeric problems, and dates that cannot be parsed.

In [ ]:
print('Exact duplicate rows:', raw.count() - raw.dropDuplicates().count())

blank_counts = [
    F.sum(F.when(F.trim(F.col(c)) == '', 1).otherwise(0)).alias(c)
    for c in raw.columns
]
raw.select(blank_counts).show()

for c in ['quantity', 'unit_price', 'discount_percent']:
    numeric = F.col(c).cast('double')
    bad_numeric = F.sum(F.when(numeric.isNull() | (numeric < 0), 1).otherwise(0)).alias('problem_rows')
    raw.select(bad_numeric).show()

parsed_date = F.coalesce(
    F.to_date('order_date', 'yyyy-MM-dd'),
    F.to_date('order_date', 'yyyy-dd-MM'),
    F.to_date('order_date', 'dd/MM/yyyy'),
    F.to_date('order_date', 'yyyy/MM/dd'),
    F.to_date('order_date', 'MM-dd-yyyy')
)
raw.select(
    F.sum(F.when(parsed_date.isNull() & (F.trim('order_date') != ''), 1).otherwise(0)).alias('unreadable_dates')
).show()

raw.groupBy('category').count().orderBy('category').show(truncate=False)


## 5. Build the Silver DataFrame

The cleaning sequence is deliberately conservative: remove exact duplicates, trim text, normalize obvious category/city/payment/status variants, validate the three business numeric fields, and parse dates using a small set of common formats.

In [ ]:
silver = raw.dropDuplicates()

# Remove accidental surrounding spaces from every column.
for c in silver.columns:
    silver = silver.withColumn(c, F.trim(F.col(c)))

# Fill missing descriptive fields rather than discarding otherwise useful rows.
text_fields = ['customer_name', 'payment_method', 'city']
for c in text_fields:
    silver = silver.withColumn(c, F.when(F.col(c) == '', None).otherwise(F.col(c)))
    silver = silver.fillna({c: 'Unknown'})

# Canonicalize the obvious case-only/category variants.
def canonicalize(col_name, pairs):
    expr = F.col(col_name)
    lowered = F.lower(expr)
    for source, target in pairs.items():
        expr = F.when(lowered == source, F.lit(target)).otherwise(expr)
    return expr

silver = silver.withColumn('category', canonicalize('category', {
    'beauty':'Beauty', 'sports':'Sports', 'electronics':'Electronics', 'clothing':'Clothing'
}))
silver = silver.withColumn('payment_method', canonicalize('payment_method', {
    'credit card':'Credit Card', 'debit card':'Debit Card', 'upi':'UPI',
    'net banking':'Net Banking', 'cash on delivery':'Cash on Delivery'
}))
silver = silver.withColumn('city', canonicalize('city', {
    'delhi':'Delhi', 'hyderabad':'Hyderabad', 'mumbai':'Mumbai', 'coimbatore':'Coimbatore',
    'jaipur':'Jaipur', 'bengaluru':'Bengaluru', 'chennai':'Chennai', 'lucknow':'Lucknow',
    'pune':'Pune', 'kolkata':'Kolkata', 'vijayawada':'Vijayawada'
}))
silver = silver.withColumn('order_status', canonicalize('order_status', {
    'delivered':'Delivered', 'shipped':'Shipped', 'processing':'Processing',
    'cancelled':'Cancelled', 'returned':'Returned'
}))

# Convert numeric fields from strings so malformed values become null.
for c in ['order_id', 'quantity', 'unit_price', 'discount_percent']:
    silver = silver.withColumn(c, F.col(c).cast('double'))

# Basic business validation. Invalid values are retained as rows and repaired below.
silver = silver.withColumn('quantity', F.when(F.col('quantity') > 0, F.col('quantity')))
silver = silver.withColumn('unit_price', F.when(F.col('unit_price') >= 0, F.col('unit_price')))
silver = silver.withColumn(
    'discount_percent',
    F.when(F.col('discount_percent').between(0, 100), F.col('discount_percent'))
)

# Median replacement for numeric gaps.
for c in ['quantity', 'unit_price', 'discount_percent']:
    q = silver.approxQuantile(c, [0.5], 0.01)
    if q:
        silver = silver.fillna({c: q[0]})

# Accept a few common date representations; genuinely invalid dates remain null.
clean_date = F.coalesce(
    F.to_date('order_date', 'yyyy-MM-dd'),
    F.to_date('order_date', 'yyyy-dd-MM'),
    F.to_date('order_date', 'dd/MM/yyyy'),
    F.to_date('order_date', 'yyyy/MM/dd'),
    F.to_date('order_date', 'MM-dd-yyyy')
)
silver = silver.withColumn('order_date', clean_date)

# Keep integer-like numeric fields as integers in the final Silver output.
silver = (silver
          .withColumn('order_id', F.col('order_id').cast('long'))
          .withColumn('quantity', F.col('quantity').cast('long'))
          .withColumn('unit_price', F.col('unit_price').cast('long'))
          .withColumn('discount_percent', F.col('discount_percent').cast('long')))

print('Silver records:', silver.count())
silver.show(5, truncate=False)


## 6. Save the Silver layer

Spark writes CSV as a directory of part files. The next cell keeps a single CSV file so the required submission has the requested `sales_silver.csv` name.

In [ ]:
import shutil

output_dir = SILVER
write_dir = Path('data_lake/silver/_spark_csv')
if write_dir.exists():
    shutil.rmtree(write_dir)
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

silver.coalesce(1).write.mode('overwrite').option('header', True).csv(str(write_dir))

part = next(write_dir.glob('part-*.csv'))
final_csv = output_dir / 'sales_silver.csv'
shutil.copyfile(part, final_csv)
shutil.rmtree(write_dir)
print('Saved:', final_csv)


## 7. Verification

In [ ]:
print('Rows before preprocessing :', raw.count())
print('Rows after preprocessing  :', silver.count())

check = spark.read.option('header', True).csv(str(SILVER / 'sales_silver.csv'))
print('Rows read back from Silver:', check.count())
check.show(5, truncate=False)

print('Preprocessing performed:')
print('- removed exact duplicate rows')
print('- trimmed text and filled missing descriptive values with Unknown')
print('- standardized obvious case inconsistencies in categorical fields')
print('- validated quantity, unit price, and discount percentage')
print('- used median values for numeric gaps created by invalid values')
print('- converted recognized date formats to yyyy-MM-dd and preserved invalid dates as null')


### Result

The Bronze copy remains unchanged. The Silver output contains the preprocessed records and is ready for submission.